# 03 · OCR benchmark: Tesseract vs PaddleOCR-VL

**Question:** which way of reading receipts gets Tarang's five fields right, how often, and at what cost in time?

| Method | What it is | Where it ran |
|---|---|---|
| `tesseract_raw` | Tesseract 5, default settings, photo as uploaded | M1 laptop CPU |
| `tesseract_pre` | Rotated upright (image audit), 3× upscale, shading removed, threshold, Tesseract `--psm 6` | M1 laptop CPU |
| `paddleocr_vl` | PaddleOCR-VL-0.9B, native `transformers`, float32, prompt `OCR:` | Kaggle T4 GPU (`kaggle/paddleocr_vl.ipynb`) |
| `paddleocr_vl_retry` | Same, but the 11 photos where generation looped (hit the token cap) are re-read with `repetition_penalty=1.05`, an automatic rule | Kaggle T4 (`kaggle/paddleocr_vl_retry.ipynb`) |
| `combined` | `paddleocr_vl_retry`, with gaps filled from `tesseract_pre` **only** where the Tesseract value passes its own check | both |

The models only **read text**. The same Python parser turns every method's text into fields, so differences come from the reading,
not from different post-processing.

**Rules that keep this honest**
* **Chronological holdout.** Parser rules and the vendor list were built on the earlier (dev) claims only. The 60 held-out claims are
  later and were never used to tune anything. A photo used on two claims never sits on both sides.
* **Clerk labels are reference labels, not ground truth.** Notebook 01 found 7 clerk GSTINs that fail the checksum; this notebook finds
  more clerk errors. Where a method disagrees with the clerk, the photo was checked and the printed value recorded
  (`reports/holdout_adjudication.csv`). Scores are reported against **both** the raw clerk labels and the adjudicated labels.
* **Exact match is exact.** The ±₹10 policy tolerance is reported separately and never mixed into field accuracy.

In [1]:
import json
import re
import subprocess
import sys
import tempfile
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from difflib import SequenceMatcher
from math import sqrt
from pathlib import Path

import pandas as pd
from PIL import Image, ImageOps

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from tarang.images import preprocess   # notebook 02's preprocessing (upscale, shading removal, threshold)

IMAGES = ROOT / "data" / "raw" / "images"
INTERIM = ROOT / "data" / "interim"
REPORTS = ROOT / "reports"
read = lambda p: pd.read_csv(p, dtype=str, keep_default_na=False)

train = read(ROOT / "data" / "processed" / "claims_train_clean.csv")
audit = read(REPORTS / "image_audit.csv")
dup = read(REPORTS / "duplicate_images.csv")
pd.set_option("display.max_colwidth", 40)
print(f"{len(train)} train claims, {(train.clerk_bill_date != '').sum()} transcribed by a clerk")

283 train claims, 277 transcribed by a clerk


## 1 · Reference labels: clerk vendor names

The clerk's vendor column has typos (`Highway King haba`), swapped words (`Route Spice Kitchen`) and city suffixes (`… (Ngp)`).
Scoring vendor against those strings would mostly measure clerk typing. So clerk labels are mapped onto **canonical shop names**
(clustered by spelling similarity; two rare spellings mapped by hand, each checked against the printed name). Every mapping is
listed below. Vendor accuracy is then "did the method identify the right shop".

In [2]:
# ---------------------------------------------------------------------------------------- vendor names
CITY_SUFFIX = {"ngp", "nagpur", "wardha", "amravati", "yavatmal", "chandrapur"}
GENERIC = {"hotel"}
# Clerk spellings that no string measure can safely join; each checked against the name printed on the photo.
VENDOR_ALIASES = {
    "sitabuldi m l parking": "Sitabuldi Multilevel Parking",   # receipt prints SITABULDI MULTILEVEL PARKING
    "shree shirdi sai petro": "Shree Sai Petroleum",           # receipt prints SHREE SAI PETROLEUM (IndianOil, Khamla)
}


def norm(s):
    return re.sub(r"[^a-z0-9]+", " ", s.lower().replace("&", " ")).strip()


def tokens_key(s):
    return " ".join(sorted(t for t in norm(s).split() if t not in CITY_SUFFIX | GENERIC))


def similarity(a, b):
    """Best of: word-order-free match, and spacing-free match (catches 'Om SaiF uel Point')."""
    ka, kb = tokens_key(a), tokens_key(b)
    sa, sb = "".join(t for t in norm(a).split() if t not in CITY_SUFFIX | GENERIC), "".join(t for t in norm(b).split() if t not in CITY_SUFFIX | GENERIC)
    return max(SequenceMatcher(None, ka, kb).ratio(), SequenceMatcher(None, sa, sb).ratio())


def canonical_vendors(labels, min_count=2, threshold=0.8):
    """Map noisy clerk vendor labels onto canonical shop names. Returns {label: canonical name}.

    Centres are names typed >= min_count times; each label joins its most similar centre if similarity >= threshold.
    The display name is the centre's most common spelling, in title case.
    """
    keyed = Counter(norm(l) for l in labels)
    spellings = {}
    for l in labels:
        spellings.setdefault(norm(l), Counter())[l] += 1
    centres = sorted((k for k, n in keyed.items() if n >= min_count), key=lambda k: -keyed[k])
    mapping = {}
    for l in set(labels):
        if norm(l) in VENDOR_ALIASES:
            mapping[l] = VENDOR_ALIASES[norm(l)]
            continue
        best = max(centres, key=lambda c: (similarity(l, c), keyed[c]))
        key = best if similarity(l, best) >= threshold else norm(l)
        mapping[l] = spellings[key].most_common(1)[0][0].title()
    return mapping

In [3]:
labelled = train[train.clerk_vendor != ""]
CANON = canonical_vendors(labelled.clerk_vendor.tolist())
mapping = pd.Series(CANON, name="canonical").rename_axis("clerk_label").reset_index()
changed = mapping[mapping.clerk_label != mapping.canonical]
print(f"{labelled.clerk_vendor.nunique()} distinct clerk spellings -> {mapping.canonical.nunique()} shops; "
      f"{changed.clerk_label.isin(labelled.clerk_vendor).sum()} spellings differ from their shop's name")
changed.sort_values(["canonical", "clerk_label"]).head(60)

117 distinct clerk spellings -> 31 shops; 87 spellings differ from their shop's name


,clerk_label,canonical
48,Dining Annapurna Hall,Annapurna Dining Hall
23,annapurna dining hall,Annapurna Dining Hall
87,Ashirwad Veg Restauraant,Ashirwad Veg Restaurant
28,Veg Ashirwad Restaurant,Ashirwad Veg Restaurant
107,CAFE COFFEE CORNER,Cafe Coffee Corner
102,GAJANAN PETROLEUM,Gajanan Petroleum
61,Gajanan Petro,Gajanan Petroleum
18,Gajanan Petroleum (Ngp),Gajanan Petroleum
115,Gajanan Pteroleum,Gajanan Petroleum
95,Petroleum Gajanan,Gajanan Petroleum


## 2 · Chronological holdout

Transcribed train claims are sorted by submission date. The latest 40% form the holdout pool; from it we keep **every hard case**
(rotated, low OCR confidence, instruction stamp) and fill to 60 by category, with a fixed seed. Earlier claims are **dev**: only they
are used to build the parser rules and the vendor list. Same-photo duplicate groups that touch dev are moved to dev entirely.

In [4]:
HANDWRITTEN = {"Highway King Dhaba", "Hotel Rajdhani Dhaba", "Punjabi Tadka Dhaba", "Sai Ram Tea & Snacks",
               "Maa Durga Cab Service", "Shree Ganesh Tours & Travels", "Royal Travels"}   # from the contact sheets


def select_holdout(train, audit, canonical, dup, n=60, pool_frac=0.4, seed=42):
    """Chronological holdout: from the latest pool_frac of transcribed train claims, keep every hard case, fill to n by category.

    Same-photo duplicate groups never straddle dev/holdout: a group touching dev is kept entirely in dev.
    """
    t = train[train.clerk_bill_date != ""].sort_values(["submitted_on", "claim_id"]).reset_index(drop=True)
    cut = int(len(t) * (1 - pool_frac))
    pool = t.iloc[cut:].copy()
    dev_ids = set(t.claim_id.iloc[:cut])
    for _, g in dup.groupby("duplicate_group"):
        if set(g.claim_id) & dev_ids:
            pool = pool[~pool.claim_id.isin(g.claim_id)]
    issues = pool.claim_id.map(audit.set_index("claim_id").issues)
    pool["hard"] = issues.str.contains("rotated|low OCR|instruction")
    pool["handwritten"] = pool.clerk_vendor.map(canonical).isin(HANDWRITTEN)
    hard = pool[pool.hard]
    rest = pool[~pool.hard]
    k = n - len(hard)
    share = (rest.category.value_counts(normalize=True) * k).round().astype(int)
    while share.sum() > k: share[share.idxmax()] -= 1
    while share.sum() < k: share[share.idxmin()] += 1
    fill = pd.concat([rest[rest.category == c].sample(m, random_state=seed) for c, m in share.items() if m > 0])
    holdout = pd.concat([hard, fill]).sort_values(["submitted_on", "claim_id"])
    return holdout, t[~t.claim_id.isin(holdout.claim_id) & ~t.claim_id.isin(pool.claim_id)], pool

In [5]:
holdout, dev, pool = select_holdout(train, audit, CANON, dup)
assert not set(holdout.claim_id) & set(dev.claim_id)
for _, g in dup.groupby("duplicate_group"):
    assert not (set(g.claim_id) & set(holdout.claim_id) and set(g.claim_id) & set(dev.claim_id)), "same photo on both sides"
issues = holdout.claim_id.map(audit.set_index("claim_id").issues)
print(f"dev {len(dev)} claims (submitted {dev.submitted_on.min()} .. {dev.submitted_on.max()}) | "
      f"holdout {len(holdout)} (submitted {holdout.submitted_on.min()} .. {holdout.submitted_on.max()})")
pd.DataFrame({
    "holdout claims": [len(holdout)],
    **{c: [(holdout.category == c).sum()] for c in sorted(holdout.category.unique())},
    "rotated": [issues.str.contains("rotated").sum()], "low OCR confidence": [issues.str.contains("low OCR").sum()],
    "instruction stamp": [issues.str.contains("instruction").sum()], "pen marks": [issues.str.contains("pen marks").sum()],
    "handwritten": [holdout.handwritten.sum()],
}).T.rename(columns={0: "count"})

dev 168 claims (submitted 2026-06-09 .. 2026-07-10) | holdout 60 (submitted 2026-07-03 .. 2026-07-18)


,count
holdout claims,60
Courier,6
Food,20
Fuel,20
Local travel,8
Lodging,3
Parking,3
rotated,8
low OCR confidence,6
instruction stamp,2


## 3 · Reading the receipts

PaddleOCR-VL ran on Kaggle (`kaggle/README.md`); its raw text is loaded from `data/interim/vlm/`. Both Tesseract baselines run here,
on every photo, and are cached in `data/interim/ocr/` (delete the files to re-run, about 3 minutes).

In [6]:
OCR_DIR = INTERIM / "ocr"
OCR_DIR.mkdir(parents=True, exist_ok=True)
ROTATION = {r.image_file: int(float(r.rotation)) for r in audit.itertuples() if r.rotation not in ("", "nan")}


def tesseract_raw(name):
    """Baseline 1: Tesseract with default settings on the photo exactly as uploaded."""
    t0 = time.perf_counter()
    text = subprocess.run(["tesseract", str(IMAGES / name), "-"], capture_output=True, text=True).stdout
    return {"file": name, "backend": "tesseract-raw", "text": text, "rotation_applied": 0, "seconds": round(time.perf_counter() - t0, 3), "error": ""}


def tesseract_pre(name):
    """Baseline 2: rotate upright (audit rotation), 3x upscale, remove shading, threshold, then Tesseract as one text block."""
    t0 = time.perf_counter()
    angle = ROTATION.get(name, 0)
    clean = preprocess(ImageOps.grayscale(Image.open(IMAGES / name))).rotate(angle, expand=True, fillcolor=255)
    with tempfile.NamedTemporaryFile(suffix=".png") as f:
        clean.save(f.name, dpi=(300, 300))
        text = subprocess.run(["tesseract", f.name, "-", "--psm", "6"], capture_output=True, text=True).stdout
    return {"file": name, "backend": "tesseract-pre", "text": text, "rotation_applied": angle, "seconds": round(time.perf_counter() - t0, 3), "error": ""}


photos = sorted(p.name for p in IMAGES.glob("*.jpg"))
for fn, out in [(tesseract_raw, "tesseract_raw.jsonl"), (tesseract_pre, "tesseract_pre.jsonl")]:
    if not (OCR_DIR / out).exists():
        with ThreadPoolExecutor(max_workers=6) as pool_:
            rows = list(pool_.map(fn, photos))
        (OCR_DIR / out).write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))

load = lambda p: pd.read_json(p, lines=True, dtype=False).set_index("file")
OUTPUTS = {
    "tesseract_raw": load(OCR_DIR / "tesseract_raw.jsonl"),
    "tesseract_pre": load(OCR_DIR / "tesseract_pre.jsonl"),
    "paddleocr_vl": load(INTERIM / "vlm" / "paddleocr_vl_outputs.jsonl"),
}
retry = load(INTERIM / "vlm" / "paddleocr_vl_retry.jsonl")
vl_retry = OUTPUTS["paddleocr_vl"].copy()
vl_retry.loc[retry.index, ["text", "new_tokens", "backend"]] = retry[["text", "new_tokens", "backend"]].values
OUTPUTS["paddleocr_vl_retry"] = vl_retry

pd.DataFrame({m: {"photos read": len(o), "errors": int((o.error != "").sum()),
                  "hit token cap": int((o.get("new_tokens", pd.Series(dtype=float)) >= 768).sum())} for m, o in OUTPUTS.items()}).T

,photos read,errors,hit token cap
tesseract_raw,467,0,0
tesseract_pre,467,0,0
paddleocr_vl,467,0,11
paddleocr_vl_retry,467,0,0


`paddleocr_vl` looped on 11 photos (repeating a line until the 768-token cap; on CLM-31087 it **invented** `Total Cost: ₹3,997,249`).
The retry rule fixed all 11. A looped output is also flagged by the parser, and its amount is never trusted.

## 4 · From text to fields

One parser for every method. Each field gets its **own status**, so the decision step can tell "read and checked" from "read but
unconfirmed" from "not found". Nothing is guessed: a missing field stays blank.

**Text clean-up.** The VLM sometimes answers in table markup (`<fcel>…<nl>`) with labels run together (`300.00Vehicle No`).

In [7]:
# ---------------------------------------------------------------------------------------- text
TABLE_TAGS = re.compile(r"<(fcel|lcel|ecel|ucel|xcel|nl)>")


FIELD_LABELS = r"(Receipt No|Bill No|Invoice No|AWB No|Ticket No|Date|Time|Pump|Nozzle|Product|Rate|Volume|Amount|Vehicle No|Mode|Thank|Save Fuel|GSTIN)"


def clean_text(text):
    """Strip the VLM's table markup (<fcel>, <nl>, ...) into plain lines; split labels that ran together ("300.00Vehicle No")."""
    text = text.replace("<nl>", "\n")
    text = TABLE_TAGS.sub("\n", text)
    text = re.sub(r"(?<=[0-9a-z.)])(?=" + FIELD_LABELS + r"\b)", "\n", text)
    return "\n".join(l.strip(" |") for l in text.splitlines())


def looped(text, new_tokens=None, cap=768):
    """True when generation ran away: hit the token cap or one line repeats many times."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    most = Counter(lines).most_common(1)[0][1] if lines else 0
    return bool((new_tokens is not None and new_tokens >= cap) or most >= 8)

**GSTIN.** A GSTIN has a fixed layout (2 digits · 5 letters · 4 digits · letter · entity · `Z` · check character). OCR confuses `O`/`0`,
`I`/`1`, `S`/`5`, so a misread is corrected **only by position** (a letter slot takes the letter, a digit slot takes the digit), and the
result is accepted **only if the official checksum passes** (`ok_normalised`). Anything else stays `invalid`; it is never "fixed" into a
valid-looking GSTIN.

In [8]:
# ---------------------------------------------------------------------------------------- GSTIN
GSTIN_CHARS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
GSTIN_PATTERN = r"\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]"
TO_DIGIT = str.maketrans({"O": "0", "D": "0", "Q": "0", "I": "1", "L": "1", "Z": "2", "S": "5", "B": "8", "G": "6", "T": "7"})
TO_LETTER = str.maketrans({"0": "O", "1": "I", "2": "Z", "5": "S", "8": "B", "6": "G", "4": "A", "7": "T"})


def gstin_is_valid(g):
    if not re.fullmatch(GSTIN_PATTERN, g):
        return False
    total = 0
    for i, ch in enumerate(g[:14]):
        p = GSTIN_CHARS.index(ch) * (2 if i % 2 else 1)
        total += p // 36 + p % 36
    return GSTIN_CHARS[(36 - total % 36) % 36] == g[14]


def gstin_by_position(raw):
    """Fix OCR letter/digit confusions using the GSTIN's fixed layout (2 digits, 5 letters, 4 digits, letter, entity, Z, check).

    Only the character class is forced by position; the result is accepted only if the official checksum passes.
    """
    g = list(raw)
    for i in range(15):
        if i in (0, 1, 7, 8, 9, 10):
            g[i] = g[i].translate(TO_DIGIT)
        elif i in (2, 3, 4, 5, 6, 11):
            g[i] = g[i].translate(TO_LETTER)
        elif i == 13:
            g[i] = "Z" if g[i] in "Z2" else g[i]
    return "".join(g)


def parse_gstin(text):
    """-> (value, status). status: ok | ok_normalised | invalid | absent. Never invents a GSTIN that fails the checksum."""
    for line in text.splitlines():
        if not re.search(r"G\s*[S5$]\s*T\s*[I1lL|]\s*N", line, re.IGNORECASE):
            continue
        after = re.split(r"G\s*[S5$]\s*T\s*[I1lL|]\s*N\s*[:.;]?", line, maxsplit=1, flags=re.IGNORECASE)[-1]
        compact = re.sub(r"[^0-9A-Za-z]", "", after.upper())
        if len(compact) < 15:
            continue
        raw = compact[:15]
        if gstin_is_valid(raw):
            return raw, "ok"
        fixed = gstin_by_position(raw)
        if gstin_is_valid(fixed):
            return fixed, "ok_normalised"
        return raw, "invalid"
    return "", "absent"

**Date.** Receipts use `08-06-2026`, `09/06/26`, `2026-06-13`, `13-Jun-26`, `08 Jun 2026`, `15.06.2026`, `9/7/2026`. All are day-first
(checked in notebook 01). Hotel bills carry check-in, check-out and invoice dates; the invoice/bill date wins.

In [9]:
# ---------------------------------------------------------------------------------------- dates
MONTHS = {m: i for i, m in enumerate(["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}
DATE_RES = [
    (re.compile(r"\b(20\d\d)[-/.](\d{1,2})[-/.](\d{1,2})\b"), "ymd"),
    (re.compile(r"\b(\d{1,2})[-/.](\d{1,2})[-/.](20\d\d|\d\d)\b"), "dmy"),
    (re.compile(r"\b(\d{1,2})[-\s]([A-Za-z]{3})[a-z]*[-\s,]+(20\d\d|\d\d)\b"), "dMy"),
]
DATE_LABEL = re.compile(r"(invoice\s*date|bill\s*date|date|dt\b|\bin\s*:)", re.IGNORECASE)


def _to_date(m, kind):
    try:
        if kind == "ymd":
            y, mo, d = int(m[1]), int(m[2]), int(m[3])
        elif kind == "dmy":
            d, mo, y = int(m[1]), int(m[2]), int(m[3])
        else:
            d, mo, y = int(m[1]), MONTHS.get(m[2][:3].lower()), int(m[3])
        y = y + 2000 if y < 100 else y
        return pd.Timestamp(year=y, month=mo, day=d)
    except (TypeError, ValueError):
        return None


def find_dates(line):
    out = []
    for rx, kind in DATE_RES:
        for m in rx.finditer(line):
            ts = _to_date(m, kind)
            if ts is not None and 2025 <= ts.year <= 2027:
                out.append((m.start(), ts))
    return [ts for _, ts in sorted(out, key=lambda x: x[0])]


def parse_date(text):
    """-> (YYYY-MM-DD, status). Prefers the bill/invoice date line over check-in/check-out and other dates."""
    lines = text.splitlines()
    ranked = []
    for i, line in enumerate(lines):
        dates = find_dates(line)
        if not dates:
            continue
        low = line.lower()
        if "invoice date" in low or "bill date" in low:
            rank = 0
        elif re.search(r"check[\s-]*(in|out)", low):
            rank = 3
        elif DATE_LABEL.search(line):
            rank = 1
        else:
            rank = 2
        ranked.append((rank, i, dates[0]))
    if not ranked:
        return "", "absent"
    ranked.sort(key=lambda x: (x[0], x[1]))
    best = ranked[0][2].strftime("%Y-%m-%d")
    others = {r[2].strftime("%Y-%m-%d") for r in ranked if r[0] == ranked[0][0]}
    return best, "ok" if len(others) == 1 else "uncertain"

**Receipt number.** Kept as text, so leading zeros survive (`003203`).

In [10]:
# ---------------------------------------------------------------------------------------- receipt number
NUMBER_LABEL = re.compile(
    r"(invoice\s*no|bill\s*no|receipt\s*no|awb\s*no|ticket\s*no|\bno)\s*[.:;|]*\s*([A-Za-z0-9][A-Za-z0-9/\-]*)", re.IGNORECASE)


def parse_receipt_number(text):
    """-> (number, status). Kept as text, leading zeros preserved."""
    priority = ["invoice", "bill", "receipt", "awb", "ticket", "no"]
    found = []
    for line in text.splitlines():
        for m in NUMBER_LABEL.finditer(line):
            label, value = m.group(1).lower(), m.group(2).strip("-/").upper()
            value = re.match(r"[A-Z0-9/\-]*\d", value).group(0) if re.search(r"\d", value) else value
            if not re.search(r"\d", value) or label.startswith("no") and len(value) < 3:
                continue
            if "vehicle" in line.lower()[:m.start() + 1] or "room" in line.lower()[:m.start() + 1]:
                continue
            kind = next(p for p in priority if label.startswith(p) or p in label)
            found.append((priority.index(kind), value))
    if not found:
        return "", "absent"
    found.sort(key=lambda x: x[0])
    return found[0][1], "ok"

**Total amount.** The riskiest field, so it is cross-checked against the receipt itself:
* restaurant and hotel bills: *sub total + CGST + SGST + round off* must equal the total;
* cab memos and hotel invoices print the amount **in words**: digits must agree with the words (this also catches `1230/-` read as `12307`);
* handwritten dhaba bills: the total is the sum of the item amounts.

`verified` means a second source on the receipt agrees; `ok` means read once; `uncertain` means the receipt contradicts itself.

In [11]:
# ---------------------------------------------------------------------------------------- amounts
NUM = r"(\d{1,3}(?:,\d{2,3})+(?:\.\d{1,2})?|\d+(?:\.\d{1,2})?)"
UNITS = {"zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
         "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18,
         "nineteen": 19, "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90}


def words_to_number(phrase):
    """'One Thousand Two Hundred Thirty only' -> 1230. None if it can't be read."""
    total, current, seen = 0, 0, False
    for w in re.findall(r"[a-z]+", phrase.lower()):
        if w in UNITS:
            current += UNITS[w]; seen = True
        elif w == "hundred":
            current = max(current, 1) * 100; seen = True
        elif w == "thousand":
            total += max(current, 1) * 1000; current = 0; seen = True
        elif w == "lakh":
            total += max(current, 1) * 100000; current = 0; seen = True
        elif w in ("rupees", "rs", "and", "only", "in", "words"):
            continue
        elif seen:
            break
    return total + current if seen else None


def to_amount(s):
    return float(s.replace(",", ""))


TOTAL_LABELS = [  # (pattern, rank): lower rank wins
    (re.compile(r"grand\s*total", re.I), 0),
    (re.compile(r"total\s*\(?\s*inr", re.I), 0),
    (re.compile(r"\btotal\b", re.I), 1),
    (re.compile(r"amount\s*(?:rs\.?|\(rs\)|:)(?!\s*in\s*words)", re.I), 2),   # "Amount Rs", "Amount (Rs):"; not the "Amount" column header
]


def _standalone_numbers(lines):
    out = []
    for i, l in enumerate(lines):
        m = re.fullmatch(r"\s*(?:rs\.?|₹)?\s*" + NUM + r"\s*(/-|7-)?\s*", l, re.I)
        if m:
            out.append((i, m.group(1), m.group(2) or ""))
    return out


def _fix_slash_dash(val, suffix, words_value):
    """'1230/-' is often read as '12307-' or '11207'. Drop the stray 7 only if the amount in words confirms it."""
    amount = to_amount(val)
    if val.endswith("7") and len(val) > 2 and (suffix == "7-" or words_value is not None):
        shorter = to_amount(val[:-1])
        if words_value is None or abs(shorter - words_value) < 0.01:
            return shorter
    return amount


SPACED = re.compile(r"\b(\d) (\d{3}(?:\.\d{1,2})?)\b")   # "1 472.00": a digit added in front, often by pen


def _fuel_amount(lines):
    """Fuel slips print rate, volume and amount: (amount line, True if rate x volume agrees within Rs 1)."""
    get = lambda label: next((float(m.group(1).replace(",", "")) for l in lines for m in [re.search(label + r"[^\d]*" + NUM, l, re.I)] if m), None)
    rate, vol, amt = get(r"rate\s*\(rs/l\)"), get(r"volume\s*\(l\)"), get(r"amount\s*\(rs\)")
    return amt, bool(rate and vol and amt and abs(rate * vol - amt) <= 1.0)


def _digits_match_words(val, words_value):
    """'11.30/-' or '12307-' next to 'One Thousand One Hundred Thirty': same digits, misplaced punctuation."""
    w = str(int(words_value))
    return re.sub(r"\D", "", val) in {w, w + "0", w + "00", w + "7"}


def parse_total(text):
    """-> (amount 'NNNN.NN', status). status: verified | words_only | ok | inconsistent | uncertain | absent.

    verified = a second source on the receipt agrees (amount in words, sub total + taxes, items sum, fuel rate x volume).
    inconsistent = the printed total contradicts the receipt's own arithmetic (a misread or an alteration); the amount
    returned is the self-consistent one.
    """
    lines = [SPACED.sub(r"\1\2", l) for l in text.splitlines()]
    words_value = None
    for i, l in enumerate(lines):
        if re.search(r"(in\s*words|rupees)", l, re.I):
            phrase = re.split(r"words\)?\s*:?", l, flags=re.I)[-1] + " " + (lines[i + 1] if i + 1 < len(lines) else "")
            words_value = words_to_number(phrase) or words_value   # the words may continue on the next line
    expected = _items_expected(lines)
    fuel_amount, fuel_ok = _fuel_amount(lines)
    candidates = []
    for i, l in enumerate(lines):
        for rx, rank in TOTAL_LABELS:
            if not rx.search(l) or re.search(r"sub\s*total|in\s*words|rate|volume", l, re.I):
                continue
            tail = rx.split(l, maxsplit=1)[-1] if rank < 2 else l
            same = re.findall(NUM + r"\s*(/-|7-)?", tail)
            if same:
                candidates.append((rank, i, *same[-1]))
            else:   # value on a nearby line: after the label, or just before a GRAND TOTAL ("914.00 / GRAND TOTAL")
                nearby = list(range(i + 1, min(i + 4, len(lines)))) + ([i - 1] if rank == 0 else [])
                for j in nearby:
                    if 0 <= j < len(lines):
                        m = re.fullmatch(r"\s*(?:rs\.?|₹)?\s*" + NUM + r"\s*(/-|7-)?\s*", lines[j], re.I)
                        if m:
                            candidates.append((rank + 0.5, i, m.group(1), m.group(2) or ""))
                            break
            break
    candidates.sort(key=lambda c: (c[0], -c[1]))
    printed = to_amount(candidates[0][2]) if candidates else None

    # fuel: the amount line is confirmed by rate x volume; a TOTAL that disagrees with it is inconsistent
    if fuel_ok:
        if printed is None or abs(printed - fuel_amount) < 0.01 or candidates[0][0] >= 2:
            return f"{fuel_amount:.2f}", "verified"
        if _extra_leading_digits(printed, fuel_amount):
            return f"{_strip_leading_digits(printed, fuel_amount):.2f}", "inconsistent"   # 536.01 -> 1536.01: digit added in front
        return f"{printed:.2f}", "inconsistent"
    # restaurants / hotels: sub total + taxes + round off
    if expected is not None:
        if printed is None or abs(printed - expected) < 0.01:
            return f"{expected:.2f}", "verified" if printed is not None or expected in _numbers(lines) else "ok"
        if _extra_leading_digits(printed, expected):
            return f"{_strip_leading_digits(printed, expected):.2f}", "inconsistent"   # 472.00 printed, "1" added by pen -> 1472.00
        return f"{printed:.2f}", "inconsistent"         # receipt contradicts itself: keep what is printed, flag it
    if words_value is not None and round(words_value, 2) in _numbers(lines) | {round(to_amount(c[2]), 2) for c in candidates}:
        return f"{words_value:.2f}", "verified"
    if words_value is not None:
        if printed is None:
            return f"{words_value:.2f}", "words_only"
        val, suffix = candidates[0][2], candidates[0][3]
        if abs(printed - words_value) < 0.01 or _digits_match_words(val, words_value):
            return f"{words_value:.2f}", "verified"
        return f"{_fix_slash_dash(val, suffix, words_value):.2f}", "uncertain"
    # handwritten bills: the largest standalone number is the sum of the others
    nums = [to_amount(v) for _, v, _ in _standalone_numbers(lines)]
    if len(nums) >= 3 and abs(max(nums) * 2 - sum(nums)) < 0.01:
        if printed is None or printed in nums:   # the label may sit next to an item price in a one-column layout
            return f"{max(nums):.2f}", "verified"
    if printed is None:
        return "", "absent"
    others = [to_amount(c[2]) for c in candidates[1:]]
    return f"{printed:.2f}", "verified" if printed in others else "ok"


def _extra_leading_digits(printed, expected):
    """True when the printed total is the expected total with digits added in front (472.00 -> 1472.00).

    Compared on whole rupees, because printed round-off can leave the computed total a few paise away (268.01 vs 268.00).
    """
    p, e = str(int(printed + 0.5)), str(int(expected + 0.5))
    return len(p) > len(e) and p.endswith(e)


def _strip_leading_digits(printed, expected):
    """1268.00 with an expected ~268 -> 268.00: the printed total without the digits added in front."""
    return round(printed % 10 ** len(str(int(expected + 0.5))), 2)


def _numbers(lines):
    return {round(to_amount(v), 2) for _, v, _ in _standalone_numbers(lines)}


def _items_expected(lines):
    """Sub total (or taxable value) + CGST + SGST + round off, as printed. None if there's no sub total."""
    def grab(label):
        for i, l in enumerate(lines):
            if re.search(label, l, re.I):
                rest = re.sub(r"[@0]?\s*[\d.]+\s*%", "", re.split(label, l, flags=re.I)[-1])   # "CGST @2.5%" / OCR "02.5%": the rate, not the tax
                same = re.findall(r"[+-]?" + NUM, rest)
                if same:
                    return float(same[-1].replace(",", ""))
                if i + 1 < len(lines):
                    nxt = re.fullmatch(r"\s*([+-]?" + NUM + r")\s*", lines[i + 1])
                    if nxt:
                        return float(nxt.group(1).replace(",", ""))
        return None
    sub = grab(r"sub\s*total|taxable\s*value")
    if sub is None:
        return None
    cgst, sgst = grab(r"cgst"), grab(r"sgst")
    if cgst is None or sgst is None or abs(cgst - sgst) > 0.01:
        return None   # CGST and SGST are always equal on these bills; unequal means the text lost a line
    round_off = grab(r"round\s*off") or 0.0
    if abs(round_off) >= 1:
        round_off = 0.0   # a real round-off is under Rs 1; a bigger value is a neighbouring line read out of place
    return round(sub + cgst + sgst + round_off, 2)

**Vendor.** PaddleOCR-VL often **skips the big header line**, so the shop name may be missing from its text. A vendor list learned from
**dev claims only** identifies the shop from other evidence: its GSTIN, a bill-number prefix (`HCP/…`), or a *signature line*
(address/phone printed on ≥2 of that shop's receipts and on no other shop's). Two lessons from building it:
* a date line was once matched to another shop's date line, so lines with dates, amounts or field labels are excluded;
* a rep's **vehicle number** had become a "signature" of Sitabuldi Parking (same rep, same car park), which forced an unseen shop
  (*Medical Square Pay & Park*) onto Sitabuldi. Vehicle numbers identify the rep, not the shop, and are excluded. An unseen shop now
  stays `unknown` with its printed name.

Each shop's **bill-number format** is learned too (`00dddd`, `HCP/dd-dd/dddd`); a number that doesn't fit is `format_mismatch`.

In [12]:
# ---------------------------------------------------------------------------------------- vendor master + matching
BRANDS = {"bpcl", "hpcl", "indianoil", "indian oil", "tax invoice", "booking receipt"}


VARIABLE_LINE = re.compile(r"\b(date|bill|receipt|invoice|awb|ticket|total|amount|rs|qty|item|steward|table|time|pump|nozzle|"
                           r"rate|volume|vehicle|veh|type|in|out|mode|name|from|to|guest|check|room|pcs|origin|dest|consignee|freight|sub|cgst|sgst)\b"
                           r"|mh\s?\d\d")   # vehicle registrations identify the rep, not the shop


def _signature_candidate(nl):
    """Lines that identify a shop (address, phone, GSTIN), not per-bill content (dates, amounts, field values)."""
    return (len(nl) >= 8 and sum(c.isalpha() for c in nl) >= 4 and nl not in BRANDS
            and not VARIABLE_LINE.search(nl) and not find_dates(nl.replace(" ", "/")))


def number_shape(no):
    """'006239' -> '00dddd', 'HCP/26-27/4594' -> 'HCP/dd-dd/dddd', 'A/6178' -> 'A/dddd'. Leading zeros are part of the shape."""
    no = no.upper()
    lead = re.match(r"0*", no).group(0)
    return lead + re.sub(r"\d", "d", no[len(lead):])


def build_vendor_master(dev):
    """Learn each shop's identifiers from DEV claims only: canonical name, GSTINs, bill-number prefixes, signature lines.

    dev: DataFrame with canonical_vendor, clerk_gstin, clerk_gstin_valid, clerk_bill_no, text (reference OCR text).
    Signature lines (address, phone) appear on >= 2 of a shop's receipts and on no other shop's.
    """
    master = {"names": sorted(dev.canonical_vendor.unique()), "gstin": {}, "prefix": {}, "lines": {}, "shapes": {}}
    for v, g in dev.groupby("canonical_vendor"):
        master["shapes"][v] = sorted({number_shape(n) for n in g.clerk_bill_no if n} | {number_shape(n)[1:] for n in g.clerk_bill_no if n.startswith("00")})
    for v, g in dev.groupby("canonical_vendor"):
        for gst in g.loc[g.clerk_gstin_valid == "True", "clerk_gstin"]:
            master["gstin"][gst] = v
    prefixes = {}
    for v, g in dev.groupby("canonical_vendor"):
        for no in g.clerk_bill_no:
            m = re.match(r"([A-Z]+/?\d*-?\d*/?|[A-Z]{2})", no.upper())
            if m and re.search(r"[A-Z]", m.group(1)):
                prefixes.setdefault(re.sub(r"\d+$", "", m.group(1)) if "/" not in m.group(1) else m.group(1).split("/")[0] + "/", set()).add(v)
    master["prefix"] = {p: next(iter(vs)) for p, vs in prefixes.items() if len(vs) == 1}
    line_vendors, line_count = {}, Counter()
    for v, g in dev.groupby("canonical_vendor"):
        seen = Counter()
        for t in g.text:
            for l in {norm(x) for x in clean_text(t).splitlines()}:
                if _signature_candidate(l):
                    seen[l] += 1
        for l, n in seen.items():
            line_vendors.setdefault(l, set()).add(v)
            line_count[(l, v)] = n
    master["lines"] = {l: next(iter(vs)) for l, vs in line_vendors.items() if len(vs) == 1 and line_count[(l, next(iter(vs)))] >= 2}
    return master


def parse_vendor(text, master, gstin=""):
    """-> (canonical vendor, status). status: ok (name read) | inferred_gstin | inferred_line | inferred_prefix | unknown | absent."""
    lines = [l for l in clean_text(text).splitlines() if l.strip()]
    best, score = None, 0.0
    for l in lines[:12]:
        for name in master["names"]:
            s = similarity(l, name)
            if s > score:
                best, score = name, s
    if score >= 0.85:
        return best, "ok"
    if gstin and gstin in master["gstin"]:
        return master["gstin"][gstin], "inferred_gstin"
    for l in lines:
        nl = norm(l)
        if nl in master["lines"]:
            return master["lines"][nl], "inferred_line"
    for l in lines:
        nl = norm(l)
        if _signature_candidate(nl):
            for sig, v in master["lines"].items():
                if SequenceMatcher(None, nl, sig).ratio() >= 0.9:
                    return v, "inferred_line"
    number, _ = parse_receipt_number(text)
    for p, v in master["prefix"].items():
        if number.startswith(p):
            return v, "inferred_prefix"
    for l in lines[:4]:
        if re.fullmatch(r"[A-Za-z&.' ]{6,}", l.strip()) and norm(l) not in BRANDS:
            return l.strip().title(), "unknown"
    return "", "absent"


def parse_receipt(text, master, new_tokens=None):
    """All five fields from one OCR/VLM text. Every field gets its own status; nothing is guessed."""
    text = clean_text(text)
    gstin, gstin_status = parse_gstin(text)
    vendor, vendor_status = parse_vendor(text, master, gstin if gstin_status.startswith("ok") else "")
    date, date_status = parse_date(text)
    number, number_status = parse_receipt_number(text)
    shapes = master["shapes"].get(vendor)
    if number and shapes:   # the shop's own numbering format, learned from dev (clerk truncations allowed one leading zero less)
        number_status = "ok" if number_shape(number) in shapes else "format_mismatch"
    total, total_status = parse_total(text)
    if looped(text, new_tokens):
        total_status = "uncertain" if total else total_status   # runaway generation can invent amounts
    return {"vendor": vendor, "vendor_status": vendor_status, "receipt_date": date, "date_status": date_status,
            "receipt_number": number, "number_status": number_status, "gstin": gstin if gstin_status != "invalid" else "",
            "gstin_read": gstin, "gstin_status": gstin_status, "total_amount": total, "total_status": total_status,
            "looped": looped(text, new_tokens)}


FIELDS = [("vendor", "vendor_status"), ("receipt_date", "date_status"), ("receipt_number", "number_status"),
          ("gstin_read", "gstin_status"), ("total_amount", "total_status")]
TRUSTED = {"vendor_status": {"ok"}, "date_status": {"ok"}, "number_status": {"ok"},  # number "ok" = fits the shop's format
           "gstin_status": {"ok", "ok_normalised"}, "total_status": {"verified"}}


def combine(primary, fallback):
    """Primary (VLM) fields, with gaps filled from the fallback (Tesseract) only where the fallback value passes its own check."""
    out = dict(primary)
    for field, status in FIELDS:
        if primary[status] in ("absent", "uncertain", "unknown", "invalid", "format_mismatch") and fallback[status] in TRUSTED[status]:
            out[field], out[status] = fallback[field], fallback[status] + "_fallback"
    out["gstin"] = out["gstin_read"] if out["gstin_status"].startswith("ok") else ""
    return out

In [13]:
dev_text = dev.assign(canonical_vendor=dev.clerk_vendor.map(CANON), text=dev.image_file.map(OUTPUTS["paddleocr_vl"].text))
MASTER = build_vendor_master(dev_text)
print(f"vendor list from {len(dev)} dev claims: {len(MASTER['names'])} shops, {len(MASTER['gstin'])} GSTINs, "
      f"{len(MASTER['prefix'])} bill-number prefixes, {len(MASTER['lines'])} signature lines")
print("prefixes:", MASTER["prefix"])

vendor list from 168 dev claims: 29 shops, 16 GSTINs, 8 bill-number prefixes, 57 signature lines
prefixes: {'HCP/': 'Hotel Centre Point Inn', 'HGE/': 'Hotel Gulmohar Executive', 'HE/': 'Hotel Gulmohar Executive', 'HKC/': 'Hotel Krishna Comforts', 'HSR/': 'Hotel Shubham Residency', 'PP': 'Prompt Parcel Services', 'SL': 'Speedlink Couriers', 'VX': 'Vidarbha Express Courier'}


## 5 · Parse every method on the holdout

In [14]:
def parse_method(name, files):
    out = OUTPUTS[name]
    return {f: parse_receipt(out.loc[f].text, MASTER, out.loc[f].get("new_tokens")) for f in files}

files = holdout.image_file.tolist()
PARSED = {m: parse_method(m, files) for m in OUTPUTS}
PARSED["combined"] = {f: combine(PARSED["paddleocr_vl_retry"][f], PARSED["tesseract_pre"][f]) for f in files}
METHODS = ["tesseract_raw", "tesseract_pre", "paddleocr_vl", "paddleocr_vl_retry", "combined"]

example = holdout.iloc[0]
pd.DataFrame({m: PARSED[m][example.image_file] for m in METHODS}).loc[
    ["vendor", "vendor_status", "receipt_date", "receipt_number", "number_status", "gstin", "gstin_status", "total_amount", "total_status"]]

,tesseract_raw,tesseract_pre,paddleocr_vl,paddleocr_vl_retry,combined
vendor,Sandwich,Spice Route Kitchen,Spice Route Kitchen,Spice Route Kitchen,Spice Route Kitchen
vendor_status,unknown,ok,inferred_gstin,inferred_gstin,inferred_gstin
receipt_date,,,2026-06-24,2026-06-24,2026-06-24
receipt_number,,5968,5968,5968,5968
number_status,absent,ok,ok,ok,ok
gstin,,,27NCRPN1784A3Z1,27NCRPN1784A3Z1,27NCRPN1784A3Z1
gstin_status,absent,absent,ok,ok,ok
total_amount,,1008.00,1008.00,1008.00,1008.00
total_status,absent,ok,verified,verified,verified


## 6 · Adjudication: when the clerk and the methods disagree

For every holdout field where PaddleOCR-VL or the combined method gave a **different value** from the clerk, the photo was checked and
the printed value recorded. Fields where the VLM and the clerk agree are accepted (two independent readings). Blank predictions are
plain misses and need no ruling.

Done by the author by eye, while building this notebook; recorded so anyone can re-check it.

In [15]:
ADJUDICATION = [
    # claim, field, clerk typed, printed on the receipt, who was right, note
    ("CLM-30807", "receipt_number", "598", "5968", "model", "Bill No: 5968; clerk dropped a digit"),
    ("CLM-30814", "total_amount", "279.00", "270.00", "model", "handwritten 150 + 120 = 270"),
    ("CLM-30820", "total_amount", "259.00", "250.00", "model", "Amount (Rs) 250.00 and TOTAL 250.00"),
    ("CLM-30824", "gstin", "27DPAFU412UX1ZM", "27DPAFU4120X1ZM", "model", "clerk value fails checksum (notebook 01)"),
    ("CLM-30846", "receipt_number", "6185", "6185", "clerk", "VLM text missed the bill number"),
    ("CLM-30868", "total_amount", "1130.00", "1130.00", "clerk", "handwritten 1130/-, model read 11307"),
    ("CLM-30883", "receipt_date", "2026-07-03", "2026-07-02", "model", "Date: 2026-07-02 20:20"),
    ("CLM-30919", "total_amount", "800.00", "300.00", "model", "TOTAL's first digit overwritten in pen (300 -> 800); "
                                                              "Amount (Rs) 300.00 and 104.64 x 2.87 = 300; receipt also carries the 'approve it' stamp"),
    ("CLM-30920", "receipt_number", "00638", "001638", "model", "Receipt No: 001638; clerk dropped a digit"),
    ("CLM-30923", "receipt_number", "007012", "007012", "clerk", "model read 067012 (flagged format_mismatch)"),
    ("CLM-30925", "receipt_number", "008732", "008732", "clerk", "blurred photo; model read 668732 (flagged format_mismatch)"),
    ("CLM-30940", "gstin", "", "27GFZCR8659B2Z4", "model", "GSTIN printed in header; clerk left it blank"),
    ("CLM-30950", "receipt_date", "2026-05-07", "2026-07-05", "model", "Date: 05-07-2026; clerk swapped day and month"),
    ("CLM-30978", "receipt_number", "006239", "006239", "clerk", "Tesseract read 906239 (flagged format_mismatch)"),
    ("CLM-30982", "gstin", "", "27OILFO1994Z2ZV", "model", "GSTIN printed; clerk left it blank"),
    ("CLM-30983", "gstin", "", "27YUFFZ9311K2ZV", "model", "GSTIN printed; clerk left it blank"),
    ("CLM-31009", "total_amount", "1536.01", "536.01", "model", "blue pen '1' added in front of TOTAL 536.01; Amount (Rs) 536.01 = "
                                                              "104.69 x 5.12; clerk rejected it as 'bill looks overwritten'"),
    ("CLM-31015", "receipt_date", "2026-07-09", "2026-07-09", "clerk", "Invoice Date 09 Jul 2026; model took check-in 07 Jul"),
    ("CLM-31034", "gstin", "27ODUFJ1789K3ZU", "27ODUFJ1789K3ZU", "clerk", "agrees with combined method"),
]
adj = pd.DataFrame(ADJUDICATION, columns=["claim_id", "field", "clerk_value", "printed_value", "correct", "note"])
adj.to_csv(REPORTS / "holdout_adjudication.csv", index=False)
print(f"{len(adj)} fields checked on the photo: clerk right {(adj.correct == 'clerk').sum()}, clerk wrong {(adj.correct == 'model').sum()}")
adj

19 fields checked on the photo: clerk right 7, clerk wrong 12


,claim_id,field,clerk_value,printed_value,correct,note
0,CLM-30807,receipt_number,598,5968,model,Bill No: 5968; clerk dropped a digit
1,CLM-30814,total_amount,279.00,270.00,model,handwritten 150 + 120 = 270
2,CLM-30820,total_amount,259.00,250.00,model,Amount (Rs) 250.00 and TOTAL 250.00
3,CLM-30824,gstin,27DPAFU412UX1ZM,27DPAFU4120X1ZM,model,clerk value fails checksum (notebook...
4,CLM-30846,receipt_number,6185,6185,clerk,VLM text missed the bill number
5,CLM-30868,total_amount,1130.00,1130.00,clerk,"handwritten 1130/-, model read 11307"
6,CLM-30883,receipt_date,2026-07-03,2026-07-02,model,Date: 2026-07-02 20:20
7,CLM-30919,total_amount,800.00,300.00,model,TOTAL's first digit overwritten in p...
8,CLM-30920,receipt_number,00638,001638,model,Receipt No: 001638; clerk dropped a ...
9,CLM-30923,receipt_number,007012,007012,clerk,model read 067012 (flagged format_mi...


**The clerks made 12 errors in the 300 holdout fields (96% field accuracy).** Dropped digits in bill numbers, day/month swapped,
GSTINs left blank, wrong amounts, and two **pen-altered totals typed as altered**: CLM-30919 (300 -> 800, approved) and
CLM-31009 (536.01 -> 1536.01, rejected). For an altered bill the reference is the original printed amount. Today's manual process is itself
not above Nilesh's 95% bar for every field.

## 7 · Scoring

| Field | Rule |
|---|---|
| Vendor | normalised exact match against the canonical shop name |
| Date | exact ISO `YYYY-MM-DD` |
| Receipt number | exact after trimming and upper-casing; **leading zeros must match** |
| GSTIN | exact. Clerk GSTINs that fail the checksum are **excluded** (count shown). A blank reference is scored separately: correct only if the method also returns blank |
| Amount | exact to the paisa. **Within ±₹10 is reported separately** and never counted as correct |

Also reported: **coverage** (share of fields where the method returned anything), **precision when answered** (accuracy on the fields it
did return), **complete-record** accuracy (all five right), and Wilson 95% intervals (n = 60 is small: ±~5–7 points).

In [16]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    centre, half = (p + z * z / (2 * n)) / (1 + z * z / n), z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (centre - half, centre + half)


def references(kind):
    """Per-claim reference values: the clerk's, or the clerk's with adjudicated corrections."""
    ref = holdout.set_index("claim_id")
    ref = pd.DataFrame({
        "image_file": ref.image_file, "vendor": ref.clerk_vendor.map(CANON), "receipt_date": ref.clerk_bill_date,
        "receipt_number": ref.clerk_bill_no, "gstin": ref.clerk_gstin, "gstin_valid": ref.clerk_gstin_valid,
        "total_amount": ref.clerk_bill_amount})
    if kind == "adjudicated":
        for r in adj.itertuples():
            ref.loc[r.claim_id, r.field] = r.printed_value
            if r.field == "gstin":
                ref.loc[r.claim_id, "gstin_valid"] = "True" if r.printed_value else ""
    return ref


def score(method, kind):
    ref = references(kind)
    rows = []
    for cid, r in ref.iterrows():
        p = PARSED[method][r.image_file]
        row = {"claim_id": cid}
        row["vendor"] = norm(p["vendor"]) == norm(r.vendor)
        row["receipt_date"] = p["receipt_date"] == r.receipt_date
        row["receipt_number"] = p["receipt_number"].strip().upper() == r.receipt_number.strip().upper()
        if r.gstin and r.gstin_valid == "False":
            row["gstin"] = None                                   # unreliable reference: excluded, counted below
        elif r.gstin:
            row["gstin"] = p["gstin"] == r.gstin
        else:
            row["gstin_absent"] = p["gstin"] == ""
        ref_amt = float(r.total_amount) if r.total_amount else None
        got = float(p["total_amount"]) if p["total_amount"] else None
        row["total_amount"] = got is not None and ref_amt is not None and abs(got - ref_amt) < 0.005
        row["total_within_10"] = got is not None and ref_amt is not None and abs(got - ref_amt) <= 10
        for f in ["vendor", "receipt_date", "receipt_number", "total_amount"]:
            row[f + "_answered"] = bool(p[f])
        row["gstin_answered"] = bool(p["gstin"])
        row["complete_record"] = all([row["vendor"], row["receipt_date"], row["receipt_number"], row["total_amount"],
                                      row.get("gstin", row.get("gstin_absent")) is not False])
        rows.append(row)
    return pd.DataFrame(rows)


def summarise(method, kind):
    s = score(method, kind)
    out = []
    for f in ["vendor", "receipt_date", "receipt_number", "gstin", "gstin_absent", "total_amount", "total_within_10", "complete_record"]:
        col = s[f].dropna() if f in s else pd.Series(dtype=bool)
        k, n = int(col.sum()), len(col)
        lo, hi = wilson(k, n)
        answered = s.get(f + "_answered")
        cov = answered.mean() if answered is not None else float("nan")
        prec = s.loc[answered & s[f].notna(), f].astype(bool).mean() if answered is not None and answered.any() else float("nan")
        out.append({"method": method, "reference": kind, "field": f, "n": n, "correct": k, "accuracy": k / n if n else float("nan"),
                    "ci95_low": lo, "ci95_high": hi, "coverage": cov, "precision_when_answered": prec,
                    "excluded_refs": int(s["gstin"].isna().sum() - s["gstin_absent"].notna().sum()) if f == "gstin" else 0})
    return out


results = pd.DataFrame([r for kind in ["clerk", "adjudicated"] for m in METHODS for r in summarise(m, kind)])
pivot = results[results.reference == "adjudicated"].pivot(index="method", columns="field", values="accuracy").loc[METHODS]
pivot = pivot[["vendor", "receipt_date", "receipt_number", "gstin", "gstin_absent", "total_amount", "total_within_10", "complete_record"]]
pivot.style.format("{:.1%}")

field,vendor,receipt_date,receipt_number,gstin,gstin_absent,total_amount,total_within_10,complete_record
method,,,,,,,,
tesseract_raw,60.0%,16.7%,25.0%,19.2%,100.0%,16.7%,16.7%,1.7%
tesseract_pre,91.7%,31.7%,61.7%,23.1%,100.0%,60.0%,60.0%,10.0%
paddleocr_vl,78.3%,85.0%,93.3%,76.9%,100.0%,93.3%,93.3%,60.0%
paddleocr_vl_retry,78.3%,85.0%,93.3%,76.9%,100.0%,95.0%,95.0%,61.7%
combined,98.3%,85.0%,95.0%,80.8%,100.0%,95.0%,95.0%,71.7%


**Same table against the raw clerk labels** (lower for the strong methods, because clerk errors count against them):

In [17]:
results[results.reference == "clerk"].pivot(index="method", columns="field", values="accuracy").loc[METHODS][pivot.columns].style.format("{:.1%}")

field,vendor,receipt_date,receipt_number,gstin,gstin_absent,total_amount,total_within_10,complete_record
method,,,,,,,,
tesseract_raw,60.0%,16.7%,23.3%,22.7%,100.0%,15.0%,16.7%,3.3%
tesseract_pre,91.7%,30.0%,58.3%,27.3%,100.0%,58.3%,58.3%,11.7%
paddleocr_vl,78.3%,81.7%,90.0%,72.7%,91.9%,86.7%,90.0%,48.3%
paddleocr_vl_retry,78.3%,81.7%,90.0%,72.7%,91.9%,88.3%,91.7%,50.0%
combined,98.3%,81.7%,91.7%,77.3%,91.9%,88.3%,91.7%,56.7%


**Uncertainty, coverage and precision** for the two strongest methods (adjudicated reference):

In [18]:
show = results[(results.reference == "adjudicated") & results.method.isin(["paddleocr_vl_retry", "combined"])].copy()
show["95% CI"] = show.apply(lambda r: f"{r.ci95_low:.0%}–{r.ci95_high:.0%}" if r.n else "", axis=1)
show[["method", "field", "n", "correct", "accuracy", "95% CI", "coverage", "precision_when_answered", "excluded_refs"]].style.format(
    {"accuracy": "{:.1%}", "coverage": "{:.0%}", "precision_when_answered": "{:.1%}"}, na_rep="")

,method,field,n,correct,accuracy,95% CI,coverage,precision_when_answered,excluded_refs
64,paddleocr_vl_retry,vendor,60,47,78.3%,66%–87%,78%,100.0%,0
65,paddleocr_vl_retry,receipt_date,60,51,85.0%,74%–92%,87%,98.1%,0
66,paddleocr_vl_retry,receipt_number,60,56,93.3%,84%–97%,97%,96.6%,0
67,paddleocr_vl_retry,gstin,26,20,76.9%,58%–89%,33%,100.0%,0
68,paddleocr_vl_retry,gstin_absent,34,34,100.0%,90%–100%,,,0
69,paddleocr_vl_retry,total_amount,60,57,95.0%,86%–98%,97%,98.3%,0
70,paddleocr_vl_retry,total_within_10,60,57,95.0%,86%–98%,,,0
71,paddleocr_vl_retry,complete_record,60,37,61.7%,49%–73%,,,0
72,combined,vendor,60,59,98.3%,91%–100%,98%,100.0%,0
73,combined,receipt_date,60,51,85.0%,74%–92%,87%,98.1%,0


GSTIN is scored on references with a valid checksum; on the holdout the clerk-invalid GSTIN was corrected by adjudication, so none
remain excluded here (`excluded_refs`). Across all 277 transcribed train claims, 7 are excluded (notebook 01).

## 8 · Speed

In [19]:
lat = []
for m in ["tesseract_raw", "tesseract_pre", "paddleocr_vl", "paddleocr_vl_retry"]:
    s = OUTPUTS[m].loc[files, "seconds"].astype(float)
    lat.append({"method": m, "median s/photo": s.median(), "p90 s/photo": s.quantile(0.9),
                "hardware": "M1 CPU, 1 photo at a time" if m.startswith("tess") else "Kaggle T4 GPU, batches of 8 (throughput)"})
pd.DataFrame(lat).set_index("method").round(2)

,median s/photo,p90 s/photo,hardware
method,,,
tesseract_raw,0.20,0.30,"M1 CPU, 1 photo at a time"
tesseract_pre,0.96,1.58,"M1 CPU, 1 photo at a time"
paddleocr_vl,2.83,9.04,"Kaggle T4 GPU, batches of 8 (through..."
paddleocr_vl_retry,2.83,9.04,"Kaggle T4 GPU, batches of 8 (through..."


Not like-for-like hardware: Tesseract timings are one photo on a laptop CPU; VLM timings are batch time ÷ batch size on one T4.
For the decision that matters (can it keep up with 2,400 claims a month?) both are far faster than needed: 2,400 photos at
~3–4 s is about 2–3 GPU-hours a month.

## 9 · What goes wrong

In [20]:
def errors(method):
    ref = references("adjudicated")
    rows = []
    for cid, r in ref.iterrows():
        p = PARSED[method][r.image_file]
        for f, st in [("vendor", "vendor_status"), ("receipt_date", "date_status"), ("receipt_number", "number_status"),
                      ("gstin", "gstin_status"), ("total_amount", "total_status")]:
            want = r[f]
            got = p[f]
            ok = norm(got) == norm(want) if f == "vendor" else (got.strip().upper() == want.strip().upper())
            if f == "total_amount" and got and want:
                ok = abs(float(got) - float(want)) < 0.005
            if not ok:
                rows.append({"claim_id": cid, "field": f, "reference": want, "predicted": got, "status": p[st],
                             "kind": "missed (blank)" if not got else "wrong value",
                             "flagged": p[st] not in ("ok", "verified", "ok_normalised") or p["looped"],
                             "issues": audit.set_index("claim_id").issues.get(cid, "")})
    return pd.DataFrame(rows)


err = errors("combined")
print(f"combined: {len(err)} wrong fields out of 300; {(err.kind == 'missed (blank)').sum()} blank, "
      f"{(err.kind == 'wrong value').sum()} wrong values, of which {(err[err.kind == 'wrong value'].flagged).sum()} flagged by the parser")
err.sort_values(["kind", "field"])

combined: 21 wrong fields out of 300; 17 blank, 4 wrong values, of which 3 flagged by the parser


,claim_id,field,reference,predicted,status,kind,flagged,issues
2,CLM-30846,gstin,27OILFO1994Z2ZV,,absent,missed (blank),True,pen marks
13,CLM-31015,gstin,27UVQFW9408Y1ZS,,absent,missed (blank),True,rotated 90; pen marks
14,CLM-31035,gstin,27OUUFX7743H2ZR,,absent,missed (blank),True,pen marks
15,CLM-31038,gstin,27NCRPN1784A3Z1,,absent,missed (blank),True,
19,CLM-31079,gstin,27OUUFX7743H2ZR,,absent,missed (blank),True,
0,CLM-30814,receipt_date,2026-06-28,,absent,missed (blank),True,low OCR confidence
1,CLM-30846,receipt_date,2026-07-01,,absent,missed (blank),True,pen marks
3,CLM-30868,receipt_date,2026-07-05,,absent,missed (blank),True,rotated 90; pen marks
6,CLM-30925,receipt_date,2026-07-02,,absent,missed (blank),True,low OCR confidence
9,CLM-30967,receipt_date,2026-07-10,,absent,missed (blank),True,pen marks


Reading the error list:
* **Blanks** dominate. The VLM skips some lines (handwritten dates, a few GSTIN and header lines) and the fallback can't always fill them.
  A blank is safe: it can never auto-approve a claim.
* **Wrong values** are the dangerous kind. The `flagged` column shows whether the parser's own checks (format mismatch, amount not
  verified, looped output) caught it, i.e. whether the decision step would still send the claim to a clerk.

In [21]:
unflagged = err[(err.kind == "wrong value") & ~err.flagged]
print(f"wrong values NOT flagged by any check: {len(unflagged)}")
unflagged

wrong values NOT flagged by any check: 1


,claim_id,field,reference,predicted,status,kind,flagged,issues
12,CLM-31015,receipt_date,2026-07-09,2026-07-07,ok,wrong value,False,rotated 90; pen marks


## 10 · Sanity check on the whole later pool (raw clerk labels)

The 60-claim holdout is small. As a second view, all 109 claims in the later pool (also never used for tuning) are scored against the
**unadjudicated** clerk labels. Expect these to be a little lower for the strong methods, because clerk errors count against them.

In [22]:
pool_files = pool.image_file.tolist()
pool_parsed = {m: parse_method(m, pool_files) for m in OUTPUTS}
pool_parsed["combined"] = {f: combine(pool_parsed["paddleocr_vl_retry"][f], pool_parsed["tesseract_pre"][f]) for f in pool_files}
rows = []
for m in METHODS:
    ok = Counter()
    n_g = 0
    for _, r in pool.iterrows():
        p = pool_parsed[m][r.image_file]
        ok["vendor"] += norm(p["vendor"]) == norm(CANON[r.clerk_vendor])
        ok["receipt_date"] += p["receipt_date"] == r.clerk_bill_date
        ok["receipt_number"] += p["receipt_number"].upper() == r.clerk_bill_no.upper()
        ok["total_amount"] += bool(p["total_amount"]) and abs(float(p["total_amount"]) - float(r.clerk_bill_amount)) < 0.005
        if r.clerk_gstin and r.clerk_gstin_valid == "True":
            n_g += 1
            ok["gstin"] += p["gstin"] == r.clerk_gstin
    rows.append({"method": m, **{f: ok[f] / len(pool) for f in ["vendor", "receipt_date", "receipt_number", "total_amount"]},
                 "gstin": ok["gstin"] / n_g, "n": len(pool), "n_gstin": n_g})
pd.DataFrame(rows).set_index("method").style.format({c: "{:.1%}" for c in ["vendor", "receipt_date", "receipt_number", "total_amount", "gstin"]})

,vendor,receipt_date,receipt_number,total_amount,gstin,n,n_gstin
method,,,,,,,
tesseract_raw,61.5%,22.9%,31.2%,20.2%,15.4%,109,39
tesseract_pre,94.5%,33.0%,56.9%,57.8%,20.5%,109,39
paddleocr_vl,78.9%,82.6%,89.0%,86.2%,76.9%,109,39
paddleocr_vl_retry,78.9%,82.6%,89.0%,87.2%,76.9%,109,39
combined,98.2%,84.4%,90.8%,87.2%,82.1%,109,39


## 11 · Save

In [23]:
results.to_csv(REPORTS / "ocr_benchmark.csv", index=False)
per_claim = []
for m in METHODS:
    for cid, r in references("adjudicated").iterrows():
        p = PARSED[m][r.image_file]
        per_claim.append({"method": m, "claim_id": cid, **{k: v for k, v in p.items()}})
pd.DataFrame(per_claim).to_csv(REPORTS / "ocr_benchmark_rows.csv", index=False)
err.to_csv(REPORTS / "ocr_benchmark_errors.csv", index=False)
print("wrote reports/ocr_benchmark.csv, ocr_benchmark_rows.csv, ocr_benchmark_errors.csv, holdout_adjudication.csv")

wrote reports/ocr_benchmark.csv, ocr_benchmark_rows.csv, ocr_benchmark_errors.csv, holdout_adjudication.csv


## Conclusions (holdout, n = 60, adjudicated labels)

* **Raw Tesseract is not usable** (3% of receipts fully right). **Preprocessing helps a lot** (vendor 92%), but its digits stay poor
  (amount 60%, date 32%): thermal-print digits blur into each other at this resolution.
* **PaddleOCR-VL reads numbers far better** (amount 93%, bill number 93%, date 85%) but often **skips the header**, so vendor alone is 78%.
  The retry rule for looped generations adds about 2 points on amount and removes the invented-amount failure.
* **Combined is best**: vendor 98%, bill number 95%, amount 95%, date 85%, GSTIN 81% (n = 26), all five fields right on 72% of receipts.
* **Its errors are mostly silence, not wrong answers.** 17 of 21 wrong fields are blanks; of the 4 wrong values, 3 are flagged by the
  parser's own checks (bill-number format, amount cross-check). One wrong value passed all checks: a hotel check-in date taken
  instead of the invoice date (the VLM text lost the invoice-date line).
* **Coverage and precision are separate.** Fields filled: vendor 59/60, date 52/60, bill number 59/60, amount 58/60, GSTIN 21/60
  (all 34 bills without one correctly blank). Of the fields filled, correct: 59/59, 51/52, 57/59, 57/58, 21/21. These counts include
  every filled value, whether or not its own check passed.
* **Caveat on the amount figure.** While building the decision engine (notebook 04) the amount parser was revised: tax-rate and
  layout bugs, and detection of pen-added digits. Two of those bugs were found through holdout errors (CLM-30829, CLM-31010), so the
  holdout amount accuracy (95.0%) is not an untouched estimate; the first, untouched run scored 93.3%.
* **Is the 95% bar met?** Not as "every field filled correctly on every receipt": dates and GSTINs fall short, mainly through blanks,
  and n = 60 can't prove 95% for any field (lower 95% bounds: vendor 91%, bill number 86%, amount 86%). What the numbers support
  is a narrower design for the decision step to test: **auto-approve only on fields that were filled and passed their checks**
  (filled-field counts above), and send everything else to a clerk.
* **Clerk typing is 96% accurate on these same fields** (12 errors in 300, including two pen-altered totals typed as altered). That is
  the baseline the tool should be compared with, not 100%.

In [24]:
a = results[(results.reference == "adjudicated")].set_index(["method", "field"])
best = "combined"
for f in ["vendor", "receipt_date", "receipt_number", "gstin", "total_amount", "complete_record"]:
    r = a.loc[(best, f)]
    print(f"{best:10s} {f:16s} {r.accuracy:6.1%}  (95% CI {r.ci95_low:.0%}–{r.ci95_high:.0%}, n={r.n})  "
          f"{'meets' if r.ci95_low >= 0.95 else 'below or not proven at'} 95%")

combined   vendor            98.3%  (95% CI 91%–100%, n=60)  below or not proven at 95%
combined   receipt_date      85.0%  (95% CI 74%–92%, n=60)  below or not proven at 95%
combined   receipt_number    95.0%  (95% CI 86%–98%, n=60)  below or not proven at 95%
combined   gstin             80.8%  (95% CI 62%–91%, n=26)  below or not proven at 95%
combined   total_amount      95.0%  (95% CI 86%–98%, n=60)  below or not proven at 95%
combined   complete_record   71.7%  (95% CI 59%–81%, n=60)  below or not proven at 95%
